# Convertir el modelo a una versión liviana (ONNX cuantizado)

**Para qué.** Las plataformas gratuitas que no piden tarjeta ofrecen unos **512 MB de memoria**, y el modelo BETO con PyTorch necesita bastante más. Este notebook toma el modelo que entrenaste en el notebook 06 (carpeta `modelo_reclamos/`) y genera una versión mucho más pequeña que corre **sin PyTorch**:

1. **ONNX:** un formato estándar de modelos que se ejecuta con `onnxruntime`, una librería ligera.
2. **Cuantización a enteros de 8 bits:** guarda los pesos con menos precisión para reducir el tamaño y la memoria, normalmente con una pérdida de precisión pequeña. **No lo damos por bueno sin medirlo**: el notebook compara las predicciones de la versión liviana con las del modelo original.

**Entrada:** carpeta `modelo_reclamos/` (del notebook 06). **Salida:** carpeta `modelo_onnx/` y `modelo_onnx.zip`, que se sube a un repositorio de modelo en Hugging Face (gratis) para que la página lo descargue al iniciar.

**Contenido:** 1 Importaciones · 2 Carga del modelo · 3 Exportar a ONNX · 4 Cuantizar · 5 Verificar que no cambió el resultado · 6 Guardar.

## 1. Importaciones

Cargamos PyTorch y transformers (para leer tu modelo) y `onnx` / `onnxruntime` (para exportarlo y ejecutarlo). En Colab suelen venir instalados los dos primeros; los de ONNX se instalan con la línea de `pip`.

In [1]:

%pip install -q onnx onnxruntime

import os, json, shutil, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from tokenizers import Tokenizer

print("PyTorch:", torch.__version__, "| onnxruntime:", ort.__version__)


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.11.0+cpu | onnxruntime: 1.32.0


## 2. Carga del modelo entrenado

Definimos las **mismas clases** del notebook 06 (BETO con dos cabezas: categoría y gravedad) y cargamos desde `modelo_reclamos/` los pesos que guardaste. `RUTA_CSV` es opcional: si existe, se usa en la verificación (sección 5) para comparar con reclamos reales; si no, se usan frases de ejemplo.

Si estás en Colab, sube primero la carpeta `modelo_reclamos` (o descomprime `modelo_reclamos.zip`) en el panel de archivos.

In [2]:
CARPETA_ENTRADA = "C:\\Users\\villa\\Downloads\\reclamos_sucre_app\\app\\modelos_reclamos"
SALIDA = "modelo_onnx"
RUTA_CSV = "C:\\Users\\villa\\Downloads\\reclamos_sucre_app\\reclamos_matias (1).csv"          # opcional, para la verificación
DEVICE = torch.device("cpu")

class ClasificadorReclamos(nn.Module):
    def __init__(self, nombre, n_cat=8, n_grav=3, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(nombre)
        self.drop = nn.Dropout(dropout)
        oculto = self.bert.config.hidden_size
        self.fc_cat = nn.Linear(oculto, n_cat)
        self.fc_grav = nn.Linear(oculto, n_grav)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        h = self.bert(**kwargs).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        pooled = self.drop((h * m).sum(1) / m.sum(1).clamp(min=1e-9))
        return self.fc_cat(pooled), self.fc_grav(pooled)

with open(os.path.join(CARPETA_ENTRADA, "meta.json"), encoding="utf-8") as f:
    meta = json.load(f)
tok = AutoTokenizer.from_pretrained(CARPETA_ENTRADA)
modelo = ClasificadorReclamos(CARPETA_ENTRADA, len(meta["categorias"]), len(meta["gravedades"]), meta["dropout"])
cab = torch.load(os.path.join(CARPETA_ENTRADA, "cabezas.pt"), map_location="cpu")
modelo.fc_cat.load_state_dict(cab["cat"]); modelo.fc_grav.load_state_dict(cab["grav"])
modelo.eval()

usa_tipos = "token_type_ids" in tok.model_input_names
print("Categorías:", meta["categorias"])
print("Usa token_type_ids:", usa_tipos, "| MAX_LEN:", meta["max_len"])

W1122 15:49:59.965000 14368 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Categorías: ['Recojo de basura', 'Agua y alcantarillado', 'Alumbrado público', 'Baches y vías', 'Transporte y tránsito', 'Seguridad ciudadana', 'Ruidos y medio ambiente', 'Otros']
Usa token_type_ids: True | MAX_LEN: 128


## 3. Exportar a ONNX

`torch.onnx.export` recorre el modelo con un ejemplo y escribe su "receta" en un archivo `.onnx`. Puntos clave:

- La **entrada** son los números del tokenizador (`input_ids`, `attention_mask` y, si el modelo lo usa, `token_type_ids`).
- La **salida** son los dos conjuntos de *logits* (categoría y gravedad); el `softmax` se calcula después, en la página.
- `dynamic_axes` indica que el **tamaño del lote y el largo del texto pueden variar**; sin eso, el modelo solo aceptaría exactamente el tamaño del ejemplo.
- El *dropout* queda desactivado porque el modelo está en modo evaluación.

Este archivo (`modelo_fp32.onnx`) todavía es grande; es el punto de partida de la cuantización.

In [3]:
class Exportable(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        return self.m(input_ids, attention_mask, token_type_ids)

nombres = ["input_ids", "attention_mask"] + (["token_type_ids"] if usa_tipos else [])
ejemplo = tok(["Ejemplo de reclamo para exportar el modelo"], return_tensors="pt")
args = tuple(ejemplo[n] for n in nombres)

os.makedirs(SALIDA, exist_ok=True)
FP32 = "modelo_fp32.onnx"
with torch.no_grad():
    torch.onnx.export(Exportable(modelo).eval(), args, FP32, input_names=nombres,
                      output_names=["logits_cat", "logits_grav"],
                      dynamic_axes={**{n: {0: "lote", 1: "tokens"} for n in nombres},
                                    "logits_cat": {0: "lote"}, "logits_grav": {0: "lote"}},
                      opset_version=17, dynamo=False)
print(f"{FP32}: {os.path.getsize(FP32) / 1e6:.0f} MB")

C:\Users\villa\AppData\Local\Temp\ipykernel_14368\1106771191.py:14: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(Exportable(modelo).eval(), args, FP32, input_names=nombres,
c:\Users\villa\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\masking_utils.py:244: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
c:\Users\villa\AppDa

modelo_fp32.onnx: 437 MB


## 4. Cuantizar a enteros de 8 bits

`quantize_dynamic` reemplaza los pesos de las capas de multiplicación de matrices (la parte más pesada de BERT) por **enteros de 8 bits** en lugar de decimales de 32 bits, lo que reduce el tamaño y el uso de memoria. Es *dinámica* porque las activaciones se convierten al vuelo y no hace falta un conjunto de calibración.

Puede aparecer una advertencia sobre inferencia de formas; no impide el resultado. Lo importante es la sección siguiente: **comprobar que las predicciones no cambiaron**.

In [4]:
INT8 = os.path.join(SALIDA, "modelo.int8.onnx")
quantize_dynamic(FP32, INT8, weight_type=QuantType.QInt8)
print(f"fp32: {os.path.getsize(FP32) / 1e6:.0f} MB  ->  int8: {os.path.getsize(INT8) / 1e6:.0f} MB")

fp32: 437 MB  ->  int8: 110 MB


## 5. Verificar que el resultado no cambió

Una versión más pequeña solo sirve si **decide igual**. Comparamos tres cosas sobre una muestra de reclamos (de tu CSV si existe, o frases de ejemplo):

1. **Tokenización:** la página usará la librería ligera `tokenizers` en lugar de `transformers`; comprobamos que produce **exactamente los mismos números**.
2. **Acuerdo de predicciones:** porcentaje de reclamos donde el modelo original y el liviano dan la **misma categoría** y la **misma gravedad**.
3. **Diferencia de probabilidades:** cuánto cambian, como máximo, las probabilidades.

Si el acuerdo es bajo (por ejemplo, menos de 95 %), no uses esta versión sin revisar; lo habitual es que sea muy alto.

In [5]:
if os.path.exists(RUTA_CSV):
    textos = pd.read_csv(RUTA_CSV, encoding="utf-8-sig")["texto"].dropna().astype(str)
    textos = textos.sample(min(300, len(textos)), random_state=0).tolist()
else:
    textos = ["Un cable de luz está caído frente a la escuela y echa chispas.",
              "Hace una semana no pasa el camión de basura por mi calle.",
              "El semáforo de la avenida no funciona y ya hubo un choque.",
              "No hay agua desde hace diez días y tengo un bebé en casa.",
              "El foco del poste de mi pasaje parpadea de vez en cuando.",
              "Asaltaron con cuchillo a una vecina en la esquina."]

# 1) tokenizador ligero vs. transformers
tk = Tokenizer.from_file(os.path.join(CARPETA_ENTRADA, "tokenizer.json"))
tk.enable_truncation(max_length=meta["max_len"])
mismos = sum(tk.encode(t).ids == tok(t, truncation=True, max_length=meta["max_len"])["input_ids"] for t in textos)
print(f"Tokenización idéntica: {mismos}/{len(textos)}")

# 2) y 3) predicciones: modelo original vs. ONNX int8
opc = ort.SessionOptions(); opc.intra_op_num_threads = 1
ses = ort.InferenceSession(INT8, opc, providers=["CPUExecutionProvider"])
softmax = lambda x: np.exp(x - x.max()) / np.exp(x - x.max()).sum()

igual_c = igual_g = 0; dif = 0.0
for t in textos:
    e = tk.encode(t)
    feeds = {"input_ids": np.array([e.ids], dtype=np.int64), "attention_mask": np.array([e.attention_mask], dtype=np.int64)}
    if usa_tipos:
        feeds["token_type_ids"] = np.array([e.type_ids], dtype=np.int64)
    lc_o, lg_o = ses.run(None, feeds)
    enc = tok([t], truncation=True, max_length=meta["max_len"], return_tensors="pt")
    with torch.no_grad():
        lc_t, lg_t = modelo(**{k: v for k, v in enc.items() if k in nombres})
    pc_t, pg_t = torch.softmax(lc_t, -1)[0].numpy(), torch.softmax(lg_t, -1)[0].numpy()
    pc_o, pg_o = softmax(lc_o[0]), softmax(lg_o[0])
    igual_c += pc_t.argmax() == pc_o.argmax(); igual_g += pg_t.argmax() == pg_o.argmax()
    dif = max(dif, np.abs(pc_t - pc_o).max(), np.abs(pg_t - pg_o).max())

n = len(textos)
print(f"Misma categoría : {igual_c}/{n} ({igual_c / n:.1%})")
print(f"Misma gravedad  : {igual_g}/{n} ({igual_g / n:.1%})")
print(f"Mayor diferencia de probabilidad: {dif:.3f}")

Tokenización idéntica: 300/300
Misma categoría : 300/300 (100.0%)
Misma gravedad  : 282/300 (94.0%)
Mayor diferencia de probabilidad: 0.169


## 6. Guardar la versión liviana

Dejamos en `modelo_onnx/` todo lo que necesita la página: el modelo cuantizado (`modelo.int8.onnx`), el `tokenizer.json` y el `meta.json` (con las categorías, el umbral de confianza y el largo máximo). Luego se comprime en `modelo_onnx.zip` (en Colab se descarga solo).

**Siguiente paso:** en huggingface.co crea un repositorio de **modelo** (gratis, puede ser privado) y sube estos archivos; la página los descargará al iniciar con la variable `MODEL_REPO`.

In [6]:
shutil.copy(os.path.join(CARPETA_ENTRADA, "tokenizer.json"), os.path.join(SALIDA, "tokenizer.json"))
meta_salida = {**meta, "formato": "onnx-int8", "usa_token_type_ids": usa_tipos}
with open(os.path.join(SALIDA, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta_salida, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile("modelo_onnx.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for a in sorted(os.listdir(SALIDA)):
        z.write(os.path.join(SALIDA, a), os.path.join("modelo_onnx", a))

print("Contenido de", SALIDA, ":", sorted(os.listdir(SALIDA)))
print(f"modelo_onnx.zip: {os.path.getsize('modelo_onnx.zip') / 1e6:.0f} MB")

try:
    from google.colab import files
    files.download("modelo_onnx.zip")
except ImportError:
    pass

Contenido de modelo_onnx : ['meta.json', 'modelo.int8.onnx', 'tokenizer.json']
modelo_onnx.zip: 72 MB
